# Random Projections for Linear Programming — Toy Demo - GUROBI
This notebook is a compact numerical demonstration of the workflow used in the project:

**Original LP → Random Projection → Projected LP → Solution Retrieval → APM**

We consider the standard-form LP

$$\min_x c^T x \quad \text{s.t.}\quad Ax=b,\;x\ge 0.$$

The random projection reduces the number of equality constraints from $m$ to $k<m$. The projected solution is generally not feasible for the original constraints, so we compare a single equality projection with Alternating Projection Method (APM) retrieval. The experiment is illustrative rather than a reproduction of the paper's full benchmark suite.


## Overview


In [ ]:
"""
Numerical example for the paper:
'Random Projections for Linear Programming: An Improved Retrieval Phase'

What this script does
---------------------
1. Generate synthetic feasible LP instances in standard form:
       min c^T x
       s.t. A x = b
            x >= 0
2. Solve the original LP.
3. Build a random projection matrix T and solve the projected LP:
       min c^T x
       s.t. T A x = T b
            x >= 0
4. Apply two retrieval methods:
   (a) Single projection (Eq. (5) in the paper)
   (b) APM = alternating projection method
5. Compare:
   - runtime
   - equality feasibility ||A x - b||
   - nonnegativity violation
   - objective gap
6. Plot four figures and save CSV files.


Important note
--------------
- This is a presentation-friendly synthetic experiment, not a full
  reproduction of all paper benchmarks.
- The script is designed to be readable and easy to modify.
"""



"\nNumerical example for the paper:\n'Random Projections for Linear Programming: An Improved Retrieval Phase'\n\nWhat this script does\n---------------------\n1. Generate synthetic feasible LP instances in standard form:\n       min c^T x\n       s.t. A x = b\n            x >= 0\n2. Solve the original LP.\n3. Build a random projection matrix T and solve the projected LP:\n       min c^T x\n       s.t. T A x = T b\n            x >= 0\n4. Apply two retrieval methods:\n   (a) Single projection (Eq. (5) in the paper)\n   (b) APM = alternating projection method\n5. Compare:\n   - runtime\n   - equality feasibility ||A x - b||\n   - nonnegativity violation\n   - objective gap\n6. Plot four figures and save CSV files.\n\nPackages used\n-------------\nnumpy\n    Basic array / matrix computation.\n\npandas\n    Save and aggregate experiment results in tables.\n\nmatplotlib\n    Plot the figures for the presentation.\n\nscipy.optimize.linprog\n    Solves LPs numerically. This is our LP solver.\n

## 1. Import required tools


In [ ]:
# =========================
# 1. Import required tools
# =========================

import time  # Used to measure runtime of each stage.
from pathlib import Path  # Used to create output folders and file paths.

import matplotlib.pyplot as plt  # Used to create plots.
import numpy as np  # Used for arrays, linear algebra, and random generation.
import pandas as pd  # Used to store results in DataFrames / CSV files.
from scipy.optimize import linprog  # Used to solve LPs.
from scipy.sparse.linalg import lsmr  # Used for approximate retrieval updates.




## 2. Global configuration for the whole script


In [ ]:
# ============================================
# 2. Global configuration for the whole script
# ============================================

# This random seed makes the whole experiment reproducible.
GLOBAL_SEED = 42

# This folder is where all figures and CSV files will be saved.
OUTPUT_DIR = Path.cwd() / "rp_lp_demo_outputs"

# This creates the output folder if it does not already exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)




## 3. Helper function: random projection T


In [ ]:
# ==========================================
# 3. Helper function: random projection T
# ==========================================


def achlioptas_projection_matrix(k: int, m: int, rng: np.random.Generator) -> np.ndarray:
    """
    Create a sparse-ish Achlioptas-style random projection matrix T.

    Parameters
    ----------
    k : int
        Projected dimension (number of rows of T).
    m : int
        Original constraint dimension (number of rows of A).
    rng : np.random.Generator
        Random number generator for reproducibility.

    Returns
    -------
    T : np.ndarray of shape (k, m)
        Random projection matrix.

    Why this form?
    --------------
    This is a common random projection construction:
    entries are in {+1, 0, -1} and then scaled by 1/sqrt(k).

    Compared with dense Gaussian T, this is lighter to compute and is
    still consistent with the general random-projection spirit.
    """
    # Sample each entry independently from {+1, 0, -1}.
    raw = rng.choice(
        [1.0, 0.0, -1.0],
        size=(k, m),
        p=[1.0 / 6.0, 2.0 / 3.0, 1.0 / 6.0],
    )

    # Scale by 1/sqrt(k), which is standard for random projection matrices.
    T = raw / np.sqrt(k)

    # Return the projection matrix.
    return T




## Setup

In [1]:
# =========================
# 1. Import required tools
# =========================

import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import gurobipy as gp
from gurobipy import GRB

from scipy.sparse.linalg import lsmr

## 2. Global configuration for the whole script


In [ ]:
# ============================================
# 2. Global configuration for the whole script
# ============================================

# This random seed makes the whole experiment reproducible.
GLOBAL_SEED = 42

# This folder is where all figures and CSV files will be saved.
OUTPUT_DIR = Path.cwd() / "rp_lp_demo_outputs"

# This creates the output folder if it does not already exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)




## 3. Helper function: random projection T


In [ ]:
# ==========================================
# 3. Helper function: random projection T
# ==========================================


def achlioptas_projection_matrix(k: int, m: int, rng: np.random.Generator) -> np.ndarray:
    """
    Create a sparse-ish Achlioptas-style random projection matrix T.

    Parameters
    ----------
    k : int
        Projected dimension (number of rows of T).
    m : int
        Original constraint dimension (number of rows of A).
    rng : np.random.Generator
        Random number generator for reproducibility.

    Returns
    -------
    T : np.ndarray of shape (k, m)
        Random projection matrix.

    Why this form?
    --------------
    This is a common random projection construction:
    entries are in {+1, 0, -1} and then scaled by 1/sqrt(k).

    Compared with dense Gaussian T, this is lighter to compute and is
    still consistent with the general random-projection spirit.
    """
    # Sample each entry independently from {+1, 0, -1}.
    raw = rng.choice(
        [1.0, 0.0, -1.0],
        size=(k, m),
        p=[1.0 / 6.0, 2.0 / 3.0, 1.0 / 6.0],
    )

    # Scale by 1/sqrt(k), which is standard for random projection matrices.
    T = raw / np.sqrt(k)

    # Return the projection matrix.
    return T




## 4. Helper function: build a feasible LP in standard form


In [ ]:
# =======================================================
# 4. Helper function: build a feasible LP in standard form
# =======================================================


def generate_feasible_lp(m: int, n: int, seed: int):
    """
    Generate a synthetic feasible LP:
        min c^T x
        s.t. A x = b
             x >= 0

    Construction idea
    -----------------
    1. Randomly generate a nonnegative vector x_true.
    2. Randomly generate a matrix A.
    3. Define b = A x_true.

    Then x_true is automatically feasible.

    Parameters
    ----------
    m : int
        Number of equality constraints.
    n : int
        Number of decision variables.
    seed : int
        Seed for the local RNG.

    Returns
    -------
    A : np.ndarray, shape (m, n)
        Constraint matrix.
    b : np.ndarray, shape (m,)
        Right-hand side vector.
    c : np.ndarray, shape (n,)
        Objective vector.
    x_true : np.ndarray, shape (n,)
        A known feasible point used to build b.
    upper_bound : float
        Common upper bound for all variables in the LP solver.
    """
    # Create a local RNG so each problem instance is reproducible.
    rng = np.random.default_rng(seed)

    # Randomly generate a dense matrix A.
    A = rng.standard_normal((m, n))

    # Create a sparse-ish nonnegative feasible vector x_true.
    x_true = np.zeros(n)

    # Select exactly m active coordinates to make x_true easier to interpret.
    active_idx = rng.choice(n, size=m, replace=False)

    # Assign positive values on the chosen active coordinates.
    x_true[active_idx] = rng.uniform(0.2, 2.0, size=m)

    # Define b = A x_true so that x_true is feasible by construction.
    b = A @ x_true

    # Create a nonnegative objective vector c.
    c = rng.uniform(0.0, 1.0, size=n)

    # Set a simple common upper bound for the LP variables.
    upper_bound = 5.0

    # Return everything needed for the LP.
    return A, b, c, x_true, upper_bound




## 5. Helper function: compute performance metrics


In [ ]:
# ============================================
# 5. Helper function: compute performance metrics
# ============================================


def compute_metrics(A: np.ndarray, b: np.ndarray, c: np.ndarray, x: np.ndarray) -> dict:
    """
    Compute the key metrics we want to compare across methods.

    Metrics
    -------
    objective
        c^T x

    eq_residual
        ||A x - b||_2
        This measures how badly the equality constraints are violated.

    negativity_sum
        sum_i max(-x_i, 0)
        This measures total nonnegativity violation.

    min_entry
        min_i x_i
        This tells us whether there are negative entries.

    Parameters
    ----------
    A, b, c, x : np.ndarray
        Standard LP ingredients.

    Returns
    -------
    metrics : dict
        Dictionary of scalar metrics.
    """
    # Compute the equality-constraint residual vector.
    residual_vector = A @ x - b

    # Compute the total amount of negative mass in x.
    negative_part_sum = np.sum(np.maximum(-x, 0.0))

    # Build the metric dictionary.
    metrics = {
        "objective": float(c @ x),
        "eq_residual": float(np.linalg.norm(residual_vector, ord=2)),
        "negativity_sum": float(negative_part_sum),
        "min_entry": float(np.min(x)),
    }

    # Return the metrics.
    return metrics


